# Exploring `biom` (Biological Observation Matrix)

Same spirit as `pivot.ipynb` / `new.ipynb`: poke at a 3rd-party library against the ABX workspace.

`biom` is the standard sparse sample × feature container in the microbiome world (QIIME2, MetaPhlAn, etc.).
Convention is **observations (taxa) × samples** — the *transpose* of our `AbundanceMatrix` (samples × taxa).

Goal of this notebook: see whether a BIOM `Table` is a viable carrier for our abundance matrix + the
two-DataFrame metadata (taxonomy lineage as *observation* metadata, sample metadata as *sample* metadata)
and what it gives us for free (collapse-by-rank, normalize, filter, HDF5 round-trip).

In [ ]:
import biom
from biom.table import Table
from biom.util import biom_open
import numpy as np
import pandas as pd

print("biom", biom.__version__)

## 1. Build the inputs from the ABX workspace (mirrors `new.ipynb`)

Note: the public `AbundanceMatrixBuilder.build(df)` route currently collides with the
`build(df, rank)` overload on this WIP branch, so we call `_build_from_dataframe` explicitly
to get the all-ranks matrix, then collapse to genus the normal way.

In [ ]:
from askslm import metadata, taxonomy

WS = "/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/inputs"

metadata_df = metadata.extract(f"{WS}/metadata.csv")

taxonomy_df = taxonomy.load(f"{WS}/taxonomy.csv")
resolved = taxonomy.resolve(taxonomy_df)

builder = taxonomy.AbundanceMatrixBuilder()
matrix = builder.build(resolved.df, "genus")   # AbundanceMatrix: samples x genera

print("abundance matrix (samples x genera):", matrix.counts.shape)
print("metadata rows:", metadata_df.shape)
matrix.counts.iloc[:3, :3]

## 2. Assemble the pieces a BIOM `Table` needs

* `data` — a (n_obs × n_samples) array → our matrix **transposed**
* `observation_ids` — taxa (we use the `taxon_key`, which is unique)
* `sample_ids` — sample accessions (`acc`)
* `observation_metadata` — taxonomy lineage per taxon (the standard `{'taxonomy': [...]}` shape)
* `sample_metadata` — a clean subset of metadata columns, aligned to sample order

In [ ]:
# --- data + ids (observations x samples) ---
wide = matrix.matrix                 # samples x taxa
obs_ids = list(wide.columns)         # taxon_key per taxon
sample_ids = list(wide.index)        # acc per sample
data = wide.to_numpy().T             # taxa x samples

# --- observation metadata: lineage as a greengenes-style 7-level list ---
RANKS = ["superkingdom", "phylum", "class", "order", "family", "genus", "species"]
lineage = matrix.taxa_lineage.set_index("taxon_key")
obs_md = [
    {"taxonomy": [lineage.loc[k, r] if (k in lineage.index and pd.notna(lineage.loc[k, r])) else ""
                  for r in RANKS]}
    for k in obs_ids
]

# --- sample metadata: a readable subset, aligned to sample_ids ---
SAMPLE_COLS = [
    "assay_type", "host_subject_id_sam", "instrument",
    "librarylayout", "geo_loc_name_country_calc", "collection_date_sam", "bioproject",
]
md_indexed = metadata_df.set_index("acc")
sample_md = [
    md_indexed.loc[s, SAMPLE_COLS].astype(str).to_dict() if s in md_indexed.index
    else {c: "" for c in SAMPLE_COLS}
    for s in sample_ids
]

print("data (taxa x samples):", data.shape)
print("example obs taxonomy:", obs_md[0]["taxonomy"])
print("example sample md:", sample_md[0])

In [ ]:
table = Table(
    data,
    observation_ids=obs_ids,
    sample_ids=sample_ids,
    observation_metadata=obs_md,
    sample_metadata=sample_md,
    table_id="ABX-genus",
)
print(table)
print("shape (obs, samples):", table.shape)
print("sparse density: %.4f" % (table.nnz / (table.shape[0] * table.shape[1])))

## 3. Collapse to a higher rank using observation metadata

This is the headline feature vs. raw pandas: BIOM collapses by a function of the observation
metadata. Collapsing genus → phylum is one line, no manual lineage join.

In [ ]:
PHYLUM_LEVEL = RANKS.index("phylum")  # 1

def to_phylum(obs_id, md):
    lin = md["taxonomy"]
    return lin[PHYLUM_LEVEL] or "Unclassified"

by_phylum = table.collapse(to_phylum, axis="observation", norm=False)
print("collapsed to phylum:", by_phylum.shape)
print("phyla:", list(by_phylum.ids(axis="observation"))[:10])

## 4. Filter by sample metadata

Compare against the `Cohort.filter(...)` path in `new.ipynb` — here it's a metadata predicate.

In [ ]:
CONTROL_SUBJECTS = {"CAN", "CAC", "CAM", "CAK", "CAA"}

def wgs_non_control(values, sample_id, md):
    return md["assay_type"] == "WGS" and md["host_subject_id_sam"] not in CONTROL_SUBJECTS

cohort = table.filter(wgs_non_control, axis="sample", inplace=False)
# drop taxa that are now all-zero across the kept samples
cohort = cohort.filter(lambda v, i, md: v.sum() > 0, axis="observation", inplace=False)
print("cohort (obs x samples):", cohort.shape)
assays = {md["assay_type"] for _, _, md in cohort.iter(axis="sample")}
print("assay types in cohort:", assays)

## 5. Relative abundance + back to a DataFrame

`norm(axis='sample')` turns per-sample counts into proportions, in place of our `relative_abundance`.

In [ ]:
rel = cohort.norm(axis="sample", inplace=False)
rel_df = rel.to_dataframe(dense=True)   # obs x samples
print("per-sample sums (should be ~1.0):")
print(rel_df.sum(axis=0).head())
rel_df.iloc[:5, :3]

## 6. HDF5 round-trip with `biom_open`

The native `.biom` format is HDF5 — carries data + both metadata axes in one file.

In [ ]:
import os
os.makedirs("out", exist_ok=True)
out_path = "out/abx_genus.biom"

with biom_open(out_path, "w") as fh:
    table.to_hdf5(fh, generated_by="Ask-SLM biom_explore")

with biom_open(out_path) as fh:
    reloaded = Table.from_hdf5(fh)

print("reloaded shape:", reloaded.shape)
print("metadata survived round-trip:")
print("  obs[0] taxonomy:", reloaded.metadata(axis="observation")[0]["taxonomy"])
print("  sample[0] md:", dict(reloaded.metadata(axis="sample")[0]))
print("file size: %.1f KB" % (os.path.getsize(out_path) / 1024))

## 7. Verdict / scratch

What we learned about whether BIOM earns a place next to `AbundanceMatrix`:

* **+** Sparse storage — the genus matrix is mostly zeros; density printed in §2.
* **+** `collapse(..., axis='observation')` replaces our rank-rollup join.
* **+** Single-file HDF5 carries data + taxonomy + sample metadata together.
* **−** Observations × samples is transposed vs. our convention — a constant `.T` foot-gun.
* **−** Fuses data + metadata into one object, against the two-DataFrame rule if used as the *only*
  carrier. Probably an *export/interchange* format, not the internal representation.

Scratch cell below for further poking.